# **Utility-Based Clinical Decision Experiments Using InfernoCalibNet for Personalized Diagnosis 🧠**

In [1]:
# ======================================================================================================================
# 📁 Setup: Root Directory, Paths, Parallelism, and Data
# ======================================================================================================================

library(inferno)

# Number of cores to use
parallel <- 10

# Root directory for all data
rootdir <- "../data/inferno"

# Directory with trained Inferno models
learntdir <- file.path(rootdir, "combined_new_MT")

# Load metadata
metadata <- read.csv(file.path(learntdir, "metadata.csv"))
print(paste("Loaded metadata with", nrow(metadata), "entries"))

# Load test data
testdata <- read.csv(file.path(rootdir, "calibration_test.csv"))
print(paste("Loaded test data with", nrow(testdata), "samples and", ncol(testdata), "features"))

[1] "Loaded metadata with 7 entries"
[1] "Loaded test data with 1459 samples and 12 features"


## Decision-Making Based on Probabilities and Utility Matrices

### 🩺 Clinical Context

An experiment is conducted using made-up clinical scenarios based on chest X-ray findings for pleural effusion and atelectasis. The scenarios are designed to reflect realistic decision-making situations involving hospital transfer, treatment initiation, or observation. Although synthetic, they aim to mirror challenges faced in real-world clinical applications, where uncertainty complicates management.

### 🧩 Traditional Utility Matrix Approach

In traditional methods, actions are tied to classification outcomes via a utility matrix. Decisions are based on the most probable diagnosis. Uncertainty beyond the chosen label is ignored, reducing robustness in the presence of critical low-probability risks.

### 🎯 Improved Decision Framework

The proposed method predicts full outcome probability distributions. Expected utilities for each action are calculated by weighting outcomes according to their predicted probabilities. The action with the highest expected utility is selected, keeping uncertainty central to the decision process.

### 🛡️ Clinical Advantage

Modeling decisions on expected utility rather than classification labels better reflects rational strategies under diagnostic uncertainty. The approach improves patient outcomes by balancing likelihood and severity instead of relying on simplified outcome predictions.

### ⚙️ Utility Matrix Modeling and Scoring

The utility matrix was manually constructed to reflect plausible clinical priorities.  
Actions were scored according to expected clinical benefit:

- **High scores** (close to 1.0) were assigned to actions critical for managing severe outcomes (e.g., hospital transfer for combined effusion and atelectasis).
- **Moderate scores** (0.5–0.7) were given to treatments targeting single conditions.
- **Lower scores** (0.2–0.4) were reserved for supportive care or observation when risk of harm was greater if untreated.

This structured design ensures that decisions are rewarded for matching patient needs and penalized for over- or under-treatment.

Unlike traditional CNN-based classification, where a utility matrix can only be applied *after* a fixed, single-label prediction, the present method integrates the full probability distribution **before** decision-making.  
Thus, it is not limited by the binary or categorical nature of CNN outputs and can directly optimize expected clinical benefit under uncertainty, something classification networks cannot achieve without significant external adjustment.

### 📊 Summary

| Aspect                  | Traditional Approach               | Improved Approach                   |
|--------------------------|-------------------------------------|--------------------------------------|
| Basis for decision       | Most probable outcome              | Expected utility                     |
| Treatment of uncertainty | Disregarded                        | Fully integrated                     |
| Robustness to error      | Low                                 | High                                 |
| Suitability for clinical practice | Limited                     | Strong                               |

In [ ]:
# ----------------------------------------------------------------------------------------------------------------------
# 🌟 Single Prediction and Clinical Decision for Chest X-rays (Effusion and Atelectasis)
# ----------------------------------------------------------------------------------------------------------------------

# Define predictands (outcome variables) and predictors (input features)
predictands <- c("LABEL_EFFUSION", "LABEL_ATELECTASIS")
predictors <- setdiff(metadata$name, predictands)

# Define possible outcomes
y <- setNames(expand.grid(0:1, 0:1), as.list(predictands))
outcomenames <- apply(y, 1, function(x) paste0("E", x[1], "_A", x[2]))

# Define available clinical actions
actions <- c(
  "Send_to_Hospital",       # For large effusions or complicated atelectasis
  "Start_Drainage_Treatment", # For moderate-large effusions
  "Start_Bronchodilator_Therapy", # For atelectasis predominantly
  "Supportive_Care",          # Mild cases
  "Observe_Closely"           # No severe findings
)

# Define utility matrix: rows = actions, columns = possible outcomes
# Higher scores reflect better alignment with patient benefit
utility_matrix <- matrix(
  c(
    # E0_A0 (No effusion, No atelectasis), E0_A1 (No effusion, Atelectasis), E1_A0 (Effusion, No atelectasis), E1_A1 (Effusion and Atelectasis)
    0.2, 0.3, 0.9, 0.95,    # Send_to_Hospital
    0.1, 0.2, 0.8, 0.85,    # Start_Drainage_Treatment
    0.3, 0.8, 0.4, 0.7,     # Start_Bronchodilator_Therapy
    0.6, 0.5, 0.3, 0.4,     # Supportive_Care
    0.7, 0.6, 0.2, 0.3      # Observe_Closely
  ),
  nrow = length(actions),
  byrow = TRUE
)
rownames(utility_matrix) <- actions
colnames(utility_matrix) <- outcomenames

# Patient index to evaluate
patient_idx <- 100

# Extract patient predictors and true labels
x_patient <- testdata[patient_idx, predictors, drop = FALSE]
true_labels <- testdata[patient_idx, predictands, drop = FALSE]

# Predict outcome probabilities
probs <- Pr(
  Y = y,
  X = x_patient,
  learnt = learntdir,
  parallel = parallel
)

# Calculate expected utilities for each action
expected_utilities <- utility_matrix %*% probs$values

# Decision function: choose the action maximizing expected utility
choose_max_action <- function(x) {
  sample(rep(which(x == max(x)), 2), 1)
}

# Make decision
decision_idx <- choose_max_action(expected_utilities)
final_decision <- actions[decision_idx]

# Enhanced Output for Better Readability
cat("\n========================================================\n")
cat("Single Patient Clinical Decision Report\n")
cat("========================================================\n")

# Show patient predictor values
cat("Patient Predictor Data (Features Only):\n")
print(x_patient)

# Show true labels
cat("\nTrue Labels (Ground Truth):\n")
print(true_labels)

# Show predicted probabilities for each disease state
cat("\nPredicted Probabilities for Outcomes:\n")
print(data.frame(Outcome = outcomenames, Probability = round(probs$values, 3)))

# Show expected utilities for each clinical action
cat("\nExpected Utilities for Clinical Actions:\n")
print(data.frame(Action = actions, Expected_Utility = round(as.numeric(expected_utilities), 3)))

# Show final recommended decision
cat("\nRecommended Clinical Action:\n")
cat(paste0(" • ", final_decision, "\n"))


Single Patient Clinical Decision Report
Patient Predictor Data (Features Only):
    AGE GENDER VP LOGIT_EFFUSION LOGIT_ATELECTASIS
100  27      M PA       1.243828         -1.856133

True Labels (Ground Truth):
    LABEL_EFFUSION LABEL_ATELECTASIS
100              1                 1

Predicted Probabilities for Outcomes:
  Outcome Probability
1   E0_A0       0.191
2   E1_A0       0.663
3   E0_A1       0.036
4   E1_A1       0.110

Expected Utilities for Clinical Actions:
                        Action Expected_Utility
1             Send_to_Hospital            0.374
2     Start_Drainage_Treatment            0.274
3 Start_Bronchodilator_Therapy            0.679
4              Supportive_Care            0.501
5              Observe_Closely            0.572

Recommended Clinical Action:
 • Start_Bronchodilator_Therapy


## 🩺 Interpretation of Single Patient Clinical Decision Report

### 👤 Patient Overview

The patient is a 27-year-old male with a VP chest X-ray. Predictors suggest high likelihood of pleural effusion (`LOGIT_EFFUSION = 1.24`) and low likelihood of atelectasis (`LOGIT_ATELECTASIS = -1.86`).  
True labels confirm the presence of both conditions.

### 📊 Predicted Outcome Probabilities

| Outcome | Description                | Probability |
|---------|-----------------------------|-------------|
| E0_A0   | No effusion, no atelectasis  | 19.1%       |
| E1_A0   | Effusion only                | 66.3%       |
| E0_A1   | Atelectasis only             | 3.6%        |
| E1_A1   | Effusion and atelectasis     | 11.0%       |

The model assigns the highest probability to isolated effusion, but combined pathology remains a relevant risk.

### 🎯 Expected Utilities and Decision

| Action                      | Expected Utility |
|------------------------------|------------------|
| Send_to_Hospital             | 0.374            |
| Start_Drainage_Treatment     | 0.274            |
| Start_Bronchodilator_Therapy | 0.679            |
| Supportive_Care              | 0.501            |
| Observe_Closely              | 0.572            |

**Start_Bronchodilator_Therapy** offers the highest expected utility.

### 🛡️ Advantage of the Outcome

The selected action addresses potential airway compromise from atelectasis while avoiding unnecessary hospitalization or invasive drainage procedures in a young, otherwise healthy patient.  
Integrating uncertainty into decision-making improves safety and matches clinical reasoning, unlike threshold-based decisions that ignore secondary risks.

✅ **Summary**:  
Expected utility-driven decisions prioritize patient welfare more effectively than classification alone, especially when managing uncertain or mixed pathologies.